# COG Conversion with Metadata (CLI Version)

Convert GeoTIFF files to Cloud Optimized GeoTIFF (COG) with embedded metadata tags using CLI commands.

> **Note:** This notebook uses CLI commands (`subprocess`) rather than Python library imports.
> For the Python API version, see `cog_metadata_template.ipynb`.

**CLI Tools Used:**
- `aws s3 ls / cp` - S3 file operations
- `rio cogeo create` - COG conversion
- `rio cogeo validate` - COG validation
- `gdal_edit.py` - Add metadata tags to GeoTIFF
- `gdalinfo` - Inspect file metadata

**Workflow:**
1. Configure paths and metadata
2. List source files
3. Define metadata
4. Process: download -> COG -> add metadata -> validate -> upload
5. Verify results

## Step 1: Configuration

In [ ]:
import subprocess, os, re, sys, time

# Add project root to path for shared_utils imports
sys.path.insert(0, os.path.abspath('..'))

# ========================================
# S3 Configuration
# ========================================
SOURCE_BUCKET = "nasa-disasters"
SOURCE_PREFIX = "drcs_activations_new/YOUR_PRODUCT/path"

DESTINATION_BUCKET = "nasa-disasters-staging"
DESTINATION_PREFIX = "ProgramData/YOUR_PRODUCT/Output/"  # Must end with /

# ========================================
# Metadata Mode: "auto" or "manual"
# ========================================
METADATA_MODE = "manual"

# Manual metadata (used when METADATA_MODE = "manual")
# YEAR_MONTH, HAZARD, LOCATION are auto-parsed from ACTIVATION_EVENT
MANUAL_METADATA = {
    "ACTIVATION_EVENT": "202501_Flood_CA",
    "PROCESSOR": "NASA Disasters COG Processor v1.0",
    # Add any custom key-value pairs here
}

# Auto-detection pattern (used when METADATA_MODE = "auto")
AUTO_DETECT_PATTERN = r'^(\d{6})_([A-Za-z0-9]+)_([A-Za-z0-9]+)_(.*)\.tif$'

# ========================================
# Processing Options
# ========================================
OVERWRITE = False
COMPRESSION = 'zstd'            # zstd, deflate, or lzw
COMPRESSION_LEVEL = 22          # 1-22 for zstd, 1-9 for deflate/lzw
DST_CRS = 'EPSG:4326'          # Target CRS for reprojection (None to keep original CRS)
OVERVIEW_LEVELS = 5
DELETE_AFTER_MOVE = False

print(f"Mode: {METADATA_MODE}")
print(f"Source: s3://{SOURCE_BUCKET}/{SOURCE_PREFIX}")
print(f"Destination: s3://{DESTINATION_BUCKET}/{DESTINATION_PREFIX}")
print(f"Compression: {COMPRESSION} (level {COMPRESSION_LEVEL})")
print(f"Target CRS: {DST_CRS if DST_CRS else 'Keep original (no reprojection)'}")

## Step 2: List Source Files

In [ ]:
result = subprocess.run(
    ['aws', 's3', 'ls', f's3://{SOURCE_BUCKET}/{SOURCE_PREFIX}/', '--recursive'],
    capture_output=True, text=True
)

tif_files = []
if result.returncode == 0:
    for line in result.stdout.strip().split('\n'):
        if line.strip() and line.strip().endswith('.tif'):
            parts = line.strip().split()
            key = parts[-1]
            size_gb = int(parts[2]) / (1024**3)
            filename = os.path.basename(key)
            tif_files.append(key)
            print(f"  {filename:<60} ({size_gb:.2f} GB)")
    print(f"\nFound {len(tif_files)} .tif files")
else:
    print(f"Error: {result.stderr}")

## Step 3: Define Metadata

In [ ]:
from shared_utils.cog_metadata import resolve_metadata

# Preview
print(f"Metadata mode: {METADATA_MODE}\n")
for key in tif_files[:5]:
    filename = os.path.basename(key)
    meta = resolve_metadata(filename, mode=METADATA_MODE, manual_metadata=MANUAL_METADATA,
                            pattern=AUTO_DETECT_PATTERN)
    print(f"  {filename}")
    for k, v in meta.items():
        print(f"    {k}: {v}")
    print()
if len(tif_files) > 5:
    print(f"  ... and {len(tif_files) - 5} more")

## Step 4: Process Files

For each file: download → create COG → add metadata tags → validate → upload.

In [ ]:
from datetime import datetime

results = []

for src_key in tif_files:
    filename = os.path.basename(src_key)
    dest_key = DESTINATION_PREFIX + filename
    local_input = f'/tmp/{filename}'
    local_warped = f'/tmp/warped_{filename}'
    local_cog = f'/tmp/cog_{filename}'

    print(f"Processing: {filename}")
    start = time.time()

    # Check if destination exists
    if not OVERWRITE:
        check = subprocess.run(
            ['aws', 's3', 'ls', f's3://{DESTINATION_BUCKET}/{dest_key}'],
            capture_output=True, text=True
        )
        if check.returncode == 0 and check.stdout.strip():
            print(f"  SKIPPED (exists)\n")
            results.append({'file': filename, 'status': 'skipped', 'time': 0})
            continue

    try:
        # 1. Download
        subprocess.run(
            ['aws', 's3', 'cp', f's3://{SOURCE_BUCKET}/{src_key}', local_input],
            capture_output=True, text=True, check=True
        )

        # 2. Reproject if DST_CRS is set (gdalwarp, all CPUs)
        cog_input = local_input
        if DST_CRS:
            import json as _json
            info = subprocess.run(
                ['gdalinfo', '-json', local_input], capture_output=True, text=True
            )
            src_crs = ''
            try:
                ginfo = _json.loads(info.stdout)
                src_crs = ginfo.get('coordinateSystem', {}).get('wkt', '')
            except Exception:
                pass

            if DST_CRS.upper() not in src_crs.upper():
                print(f"  Reprojecting to {DST_CRS} (all CPUs)...")
                # gdalwarp chosen over `rio warp` so we can use NUM_THREADS=ALL_CPUS.
                subprocess.run([
                    'gdalwarp',
                    '-t_srs', DST_CRS,
                    '-multi',
                    '-wo', 'NUM_THREADS=ALL_CPUS',
                    '-co', 'NUM_THREADS=ALL_CPUS',
                    '--config', 'GDAL_NUM_THREADS', 'ALL_CPUS',
                    '-overwrite',
                    local_input, local_warped,
                ], capture_output=True, text=True, check=True)
                cog_input = local_warped
            else:
                print(f"  Already in {DST_CRS}, skipping reprojection")

        # 3. Convert to COG (all CPUs)
        cmd = [
            'rio', 'cogeo', 'create', cog_input, local_cog,
            '--cog-profile', COMPRESSION,
            '--overview-level', str(OVERVIEW_LEVELS),
            '--overview-resampling', 'average',
            '--co', 'NUM_THREADS=ALL_CPUS',
        ]
        if COMPRESSION.lower() == 'zstd':
            cmd.extend(['--co', f'ZSTD_LEVEL={COMPRESSION_LEVEL}'])
        elif COMPRESSION.lower() in ('deflate', 'lzw'):
            cmd.extend(['--co', f'ZLEVEL={COMPRESSION_LEVEL}'])
        subprocess.run(cmd, capture_output=True, text=True, check=True)

        # 4. Add metadata tags with gdal_edit.py
        metadata = resolve_metadata(filename, mode=METADATA_MODE, manual_metadata=MANUAL_METADATA,
                                    pattern=AUTO_DETECT_PATTERN)
        metadata['PROCESSING_DATE'] = datetime.utcnow().isoformat()

        gdal_edit_cmd = ['gdal_edit.py']
        for key, value in metadata.items():
            gdal_edit_cmd.extend(['-mo', f'{key}={value}'])
        gdal_edit_cmd.append(local_cog)
        subprocess.run(gdal_edit_cmd, capture_output=True, text=True, check=True)

        # 5. Validate COG
        val = subprocess.run(
            ['rio', 'cogeo', 'validate', local_cog],
            capture_output=True, text=True
        )
        is_valid = val.returncode == 0
        if is_valid:
            print(f"  COG valid")
        else:
            print(f"  WARNING: COG validation issue: {val.stderr[:200]}")

        # 6. Upload
        subprocess.run(
            ['aws', 's3', 'cp', local_cog, f's3://{DESTINATION_BUCKET}/{dest_key}'],
            capture_output=True, text=True, check=True
        )

        elapsed = time.time() - start
        print(f"  SUCCESS ({elapsed:.1f}s) -> s3://{DESTINATION_BUCKET}/{dest_key}\n")
        results.append({'file': filename, 'status': 'success', 'time': elapsed, 'cog_valid': is_valid})

        # 7. Optionally delete source
        if DELETE_AFTER_MOVE:
            subprocess.run(
                ['aws', 's3', 'rm', f's3://{SOURCE_BUCKET}/{src_key}'],
                capture_output=True, text=True
            )
            print(f"  Deleted from source")

    except subprocess.CalledProcessError as e:
        elapsed = time.time() - start
        print(f"  FAILED: {e.stderr[:200] if e.stderr else str(e)}\n")
        results.append({'file': filename, 'status': 'failed', 'time': elapsed, 'error': str(e)})

    finally:
        for f in [local_input, local_warped, local_cog]:
            if os.path.exists(f):
                os.remove(f)

## Step 5: Results Summary

In [ ]:
total = len(results)
success = sum(1 for r in results if r['status'] == 'success')
failed = sum(1 for r in results if r['status'] == 'failed')
skipped = sum(1 for r in results if r['status'] == 'skipped')

print(f"Total: {total}")
print(f"  Success: {success}")
print(f"  Failed:  {failed}")
print(f"  Skipped: {skipped}")

if success > 0:
    times = [r['time'] for r in results if r['status'] == 'success']
    print(f"\nAvg time: {sum(times)/len(times):.1f}s per file")

if failed > 0:
    print("\nFailed files:")
    for r in results:
        if r['status'] == 'failed':
            print(f"  - {r['file']}: {r.get('error', 'Unknown')}")

## Step 6: Verify Metadata

Download a processed file and inspect metadata with `gdalinfo`.

In [ ]:
successful = [r for r in results if r['status'] == 'success']
if successful:
    sample = successful[0]
    dest_key = DESTINATION_PREFIX + sample['file']
    local_verify = f"/tmp/verify_{sample['file']}"

    print(f"Verifying: s3://{DESTINATION_BUCKET}/{dest_key}\n")

    # Download
    subprocess.run(
        ['aws', 's3', 'cp', f's3://{DESTINATION_BUCKET}/{dest_key}', local_verify],
        capture_output=True, text=True, check=True
    )

    # Show metadata with gdalinfo
    info = subprocess.run(
        ['gdalinfo', local_verify], capture_output=True, text=True
    )
    # Extract metadata lines
    in_metadata = False
    for line in info.stdout.split('\n'):
        if 'Metadata:' in line:
            in_metadata = True
            print(line)
        elif in_metadata and line.startswith('  '):
            print(line)
        elif in_metadata:
            in_metadata = False

    # Validate COG
    print()
    val = subprocess.run(
        ['rio', 'cogeo', 'validate', local_verify], capture_output=True, text=True
    )
    print(val.stdout or val.stderr)

    os.remove(local_verify)
else:
    print("No successful files to verify")

## Troubleshooting

- **No files found** - Check `SOURCE_PREFIX`, verify bucket permissions with `aws s3 ls`
- **Files skipped** - Set `OVERWRITE = True` in Step 1
- **`gdal_edit.py` not found** - Install GDAL: `conda install -c conda-forge gdal` or `brew install gdal`
- **`rio cogeo` not found** - Install: `pip install rio-cogeo`
- **Metadata not showing in gdalinfo** - Make sure `gdal_edit.py` ran successfully (step 3 in processing loop)
- **COG validation warnings** - Usually harmless; `gdal_edit.py` modifies in-place without breaking COG structure
- **For Python API version** - Use `cog_metadata_template.ipynb` (processes in-memory, no temp files for small data)